1.1. CURATION OF ChEMBL MIC DATABASE

MIC data curation, structural cleaning and standardization, molar concentration conversion, duplicate resolution, and pMIC calculation for *Acinetobacter baumannii* and *Staphylococcus aureus*

Compatibility: Google Colab


STEP-BY-STEP OVERVIEW

1. Install and import the required Python libraries and record software versions.
2. Select the antibacterial model: *Acinetobacter baumannii* or *Staphylococcus aureus*.
3. Upload one or multiple ChEMBL MIC datasets in CSV format.
4. Automatically concatenate multiple uploaded datasets by matching column names.
5. Audit the combined dataset for MIC records, SMILES, Standard Value, Standard Units, and concentration-unit formats.
6. Remove unusable records lacking SMILES, Standard Value, or Standard Units, as well as records with non-positive MIC values.
7. Validate the original SMILES and clean molecular structures by removing salts and counterion fragments and excluding metal-containing parent structures.
8. Generate cleaned, canonical, and isomeric SMILES representations using RDKit.
9. Detect the reported MIC units and mathematically standardize all valid MIC values to molar concentration (M), using molecular weight when required for mass-based units.
10. Identify duplicate structures using standardized canonical SMILES and retain the record with the lowest molar MIC value.
11. Calculate pMIC values from the standardized molar MIC values.
12. Organize the final curated dataset while preserving the original ChEMBL columns and adding `SMILES_Clean`, `Canonical_SMILES`, `Isomeric_SMILES`, `MIC_M`, and `pMIC`.
13. Generate the final curation and quality-control summary.
14. Export the final curated ChEMBL MIC dataset in CSV format.

In [ ]:
# ============================================================
# Step 1.
# ============================================================


!pip -q install rdkit

import io
import sys
import math
import re

import numpy as np
import pandas as pd

from google.colab import files
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit import rdBase
from rdkit import RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize

# Suppress unnecessary RDKit console messages
RDLogger.DisableLog("rdApp.*")

print("Runtime environment successfully initialized.")
print("-" * 55)
print(f"Python version : {sys.version.split()[0]}")
print(f"pandas version : {pd.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"RDKit version  : {rdBase.rdkitVersion}")

Runtime environment successfully initialized.
-------------------------------------------------------
Python version : 3.13.15
pandas version : 2.2.3
NumPy version  : 2.1.3
RDKit version  : 2026.03.5


In [ ]:
# ============================================================
# Step 2. Antibacterial model selection
# ============================================================

MODEL_OPTIONS = {
    1: {
        "name": "Acinetobacter baumannii",
        "prefix": "A_baumannii"
    },
    2: {
        "name": "Staphylococcus aureus",
        "prefix": "S_aureus"
    }
}

while True:
    try:
        print("Select the antibacterial model:")
        print("1 - Acinetobacter baumannii")
        print("2 - Staphylococcus aureus")

        model_selection = int(input("\nEnter option (1 or 2): "))

        if model_selection in MODEL_OPTIONS:
            break

        print("\nInvalid option. Please enter 1 or 2.\n")

    except ValueError:
        print("\nInvalid input. Please enter a numeric option.\n")

MODEL_NAME = MODEL_OPTIONS[model_selection]["name"]
MODEL_PREFIX = MODEL_OPTIONS[model_selection]["prefix"]

print("\nAntibacterial model selected:")
print(f"{MODEL_NAME}")

Select the antibacterial model:
1 - Acinetobacter baumannii
2 - Staphylococcus aureus

Enter option (1 or 2): 1

Antibacterial model selected:
Acinetobacter baumannii


In [ ]:
# ============================================================
# Step 3. Upload ChEMBL dataset(s)
# ============================================================

print("Upload one or multiple ChEMBL CSV files.\n")

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No files were uploaded.")

print("\nFiles successfully uploaded:")
for filename in uploaded.keys():
    print(f" - {filename}")

Upload one or multiple ChEMBL CSV files.



Saving Acinetobacter baumannii (base de datos).csv to Acinetobacter baumannii (base de datos).csv

Files successfully uploaded:
 - Acinetobacter baumannii (base de datos).csv


In [ ]:
# ============================================================
# Step 4. Automatic dataset concatenation
# ============================================================

dataframes = []

for filename, file_content in uploaded.items():

    if not filename.lower().endswith(".csv"):
        raise ValueError(
            f"Unsupported file format detected: {filename}. "
            "Please upload CSV files only."
        )

    temp_df = pd.read_csv(
        io.BytesIO(file_content),
        low_memory=False
    )

    dataframes.append(temp_df)

    print(
        f"{filename}: "
        f"{temp_df.shape[0]:,} rows x {temp_df.shape[1]} columns"
    )

# Concatenate all uploaded datasets
chembl_data = pd.concat(
    dataframes,
    ignore_index=True,
    sort=False
)

# Preserve the original ChEMBL column order
original_columns = chembl_data.columns.tolist()

print("\nDataset integration completed.")
print("-" * 55)
print(f"Number of uploaded files : {len(dataframes)}")
print(f"Total records            : {len(chembl_data):,}")
print(f"Total columns            : {chembl_data.shape[1]}")

Acinetobacter baumannii (base de datos).csv: 14,617 rows x 48 columns

Dataset integration completed.
-------------------------------------------------------
Number of uploaded files : 1
Total records            : 14,617
Total columns            : 48


In [ ]:
# ============================================================
# Step 5. Initial dataset audit
# ============================================================

REQUIRED_COLUMNS = [
    "Molecule ID",
    "Molecule Name",
    "Molecular Weight",
    "Smiles",
    "Standard Type",
    "Standard Relation",
    "Standard Value",
    "Standard Units"
]

missing_required_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in chembl_data.columns
]

if missing_required_columns:
    raise ValueError(
        "The following required ChEMBL columns are missing:\n"
        + "\n".join(missing_required_columns)
    )

# Convert Standard Value temporarily for audit purposes only
standard_value_numeric = pd.to_numeric(
    chembl_data["Standard Value"],
    errors="coerce"
)

missing_smiles = (
    chembl_data["Smiles"].isna()
    | chembl_data["Smiles"].astype(str).str.strip().eq("")
)

missing_standard_value = standard_value_numeric.isna()

nonpositive_standard_value = (
    standard_value_numeric.notna()
    & (standard_value_numeric <= 0)
)

missing_standard_units = (
    chembl_data["Standard Units"].isna()
    | chembl_data["Standard Units"].astype(str).str.strip().eq("")
)

print("INITIAL DATASET AUDIT")
print("=" * 55)

print(f"Total records                  : {len(chembl_data):,}")
print(f"Missing SMILES                 : {missing_smiles.sum():,}")
print(f"Missing Standard Value         : {missing_standard_value.sum():,}")
print(f"Standard Value <= 0            : {nonpositive_standard_value.sum():,}")
print(f"Missing Standard Units         : {missing_standard_units.sum():,}")

print("\nBioactivity types detected:")
print(
    chembl_data["Standard Type"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nStandard Units detected:")
print(
    chembl_data["Standard Units"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nStandard Relations detected:")
print(
    chembl_data["Standard Relation"]
    .value_counts(dropna=False)
    .to_string()
)

INITIAL DATASET AUDIT
Total records                  : 14,617
Missing SMILES                 : 237
Missing Standard Value         : 574
Standard Value <= 0            : 1
Missing Standard Units         : 574

Bioactivity types detected:
Standard Type
MIC    14432
NaN      185

Standard Units detected:
Standard Units
ug.mL-1    10686
nM          3357
NaN          574

Standard Relations detected:
Standard Relation
'='     8607
'>'     4958
NaN      574
'>='     238
'<='     198
'<'       42


In [ ]:
# ============================================================
# Step 6. Removal of unusable MIC records
# ============================================================

curated = chembl_data.copy()

audit = {}

audit["Initial records"] = len(curated)

# ------------------------------------------------------------
# 6.1 Retain MIC records
# ------------------------------------------------------------

mic_mask = (
    curated["Standard Type"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("MIC")
)

audit["Non-MIC records removed"] = int((~mic_mask).sum())

curated = curated.loc[mic_mask].copy()

# ------------------------------------------------------------
# 6.2 Remove records without SMILES
# ------------------------------------------------------------

missing_smiles_mask = (
    curated["Smiles"].isna()
    | curated["Smiles"].astype(str).str.strip().eq("")
)

audit["Missing SMILES removed"] = int(
    missing_smiles_mask.sum()
)

curated = curated.loc[~missing_smiles_mask].copy()

# ------------------------------------------------------------
# 6.3 Convert Standard Value internally to numeric form
# ------------------------------------------------------------

curated["_Standard_Value_Numeric"] = pd.to_numeric(
    curated["Standard Value"],
    errors="coerce"
)

# Remove missing/non-numeric Standard Value
missing_value_mask = (
    curated["_Standard_Value_Numeric"].isna()
)

audit["Missing Standard Value removed"] = int(
    missing_value_mask.sum()
)

curated = curated.loc[~missing_value_mask].copy()

# ------------------------------------------------------------
# 6.4 Remove zero or negative MIC values
# ------------------------------------------------------------

nonpositive_mask = (
    curated["_Standard_Value_Numeric"] <= 0
)

audit["Non-positive Standard Value removed"] = int(
    nonpositive_mask.sum()
)

curated = curated.loc[~nonpositive_mask].copy()

# ------------------------------------------------------------
# 6.5 Remove records without Standard Units
# ------------------------------------------------------------

missing_units_mask = (
    curated["Standard Units"].isna()
    | curated["Standard Units"]
        .astype(str)
        .str.strip()
        .eq("")
)

audit["Missing Standard Units removed"] = int(
    missing_units_mask.sum()
)

curated = curated.loc[~missing_units_mask].copy()

print("INITIAL BIOACTIVITY FILTERING COMPLETED")
print("=" * 55)

for key, value in audit.items():
    print(f"{key:<38}: {value:,}")

print("-" * 55)
print(f"Records remaining: {len(curated):,}")

INITIAL BIOACTIVITY FILTERING COMPLETED
Initial records                       : 14,617
Non-MIC records removed               : 185
Missing SMILES removed                : 52
Missing Standard Value removed        : 389
Non-positive Standard Value removed   : 1
Missing Standard Units removed        : 0
-------------------------------------------------------
Records remaining: 13,990


In [ ]:
# ============================================================
# Step 7. SMILES validation and structural cleaning
# ============================================================

# Atomic numbers classified here as metallic elements.
# Metal counterions present as disconnected fragments are removed
# before this check. Metal atoms remaining in the principal molecular
# structure cause the record to be excluded.

METAL_ATOMIC_NUMBERS = (
    {
        # Alkali metals
        3, 11, 19, 37, 55, 87,

        # Alkaline-earth metals
        4, 12, 20, 38, 56, 88,

        # Selected post-transition metals
        13, 31, 49, 50, 81, 82, 83, 84
    }
    | set(range(21, 31))     # First-row transition metals
    | set(range(39, 49))     # Second-row transition metals
    | set(range(57, 81))     # Lanthanides + third-row transition metals
    | set(range(89, 113))    # Actinides + heavier metals
)

# Prefer the largest organic fragment when multiple disconnected
# molecular components are present.
fragment_chooser = rdMolStandardize.LargestFragmentChooser(
    preferOrganic=True
)


def clean_smiles_structure(smiles):
    """
    Parse an original SMILES string, retain the principal molecular
    fragment, remove disconnected salt/counterion fragments, and
    exclude structures containing metals in the retained parent.

    Returns
    -------
    clean_smiles : str or None
        Cleaned parent SMILES representation.
    status : str
        Processing status.
    """

    if pd.isna(smiles):
        return None, "missing_smiles"

    smiles = str(smiles).strip()

    if smiles == "":
        return None, "missing_smiles"

    # Parse SMILES
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None, "invalid_smiles"

    try:
        # Retain the principal molecular component
        parent_mol = fragment_chooser.choose(mol)

    except Exception:
        return None, "fragment_cleaning_failed"

    # Detect metals remaining in the retained molecular parent
    contains_metal = any(
        atom.GetAtomicNum() in METAL_ATOMIC_NUMBERS
        for atom in parent_mol.GetAtoms()
    )

    if contains_metal:
        return None, "metal_in_parent_structure"

    # Generate a cleaned SMILES while retaining available stereochemistry
    clean_smiles = Chem.MolToSmiles(
        parent_mol,
        canonical=False,
        isomericSmiles=True
    )

    return clean_smiles, "valid"


print("Processing molecular structures...")

structure_results = [
    clean_smiles_structure(smiles)
    for smiles in tqdm(
        curated["Smiles"],
        total=len(curated)
    )
]

structure_results_df = pd.DataFrame(
    structure_results,
    columns=[
        "SMILES_Clean",
        "_Structure_Status"
    ],
    index=curated.index
)

curated = pd.concat(
    [curated, structure_results_df],
    axis=1
)

# Count processing outcomes
structure_status_counts = (
    curated["_Structure_Status"]
    .value_counts()
)

print("\nSTRUCTURAL CLEANING SUMMARY")
print("=" * 55)
print(structure_status_counts.to_string())

# Keep successfully processed structures only
valid_structure_mask = (
    curated["_Structure_Status"] == "valid"
)

audit["Invalid/metal structures removed"] = int(
    (~valid_structure_mask).sum()
)

curated = curated.loc[
    valid_structure_mask
].copy()

print("-" * 55)
print(f"Structures remaining: {len(curated):,}")

Processing molecular structures...


  0%|          | 0/13990 [00:00<?, ?it/s]


STRUCTURAL CLEANING SUMMARY
_Structure_Status
valid    13990
-------------------------------------------------------
Structures remaining: 13,990


In [ ]:
# ============================================================
# Step 8. Generate canonical and isomeric SMILES
# ============================================================

def generate_standardized_smiles(clean_smiles):
    """
    Generate canonical and canonical-isomeric SMILES from a
    cleaned molecular structure.
    """

    mol = Chem.MolFromSmiles(clean_smiles)

    if mol is None:
        return None, None

    canonical_smiles = Chem.MolToSmiles(
        mol,
        canonical=True,
        isomericSmiles=False
    )

    isomeric_smiles = Chem.MolToSmiles(
        mol,
        canonical=True,
        isomericSmiles=True
    )

    return canonical_smiles, isomeric_smiles


print("Generating standardized SMILES representations...")

standardized_results = [
    generate_standardized_smiles(smiles)
    for smiles in tqdm(
        curated["SMILES_Clean"],
        total=len(curated)
    )
]

standardized_df = pd.DataFrame(
    standardized_results,
    columns=[
        "Canonical_SMILES",
        "Isomeric_SMILES"
    ],
    index=curated.index
)

curated = pd.concat(
    [curated, standardized_df],
    axis=1
)

# Remove any unexpected failed standardizations
failed_standardization = (
    curated["Canonical_SMILES"].isna()
    | curated["Isomeric_SMILES"].isna()
)

audit["SMILES standardization failures removed"] = int(
    failed_standardization.sum()
)

curated = curated.loc[
    ~failed_standardization
].copy()

print("\nSMILES standardization completed.")
print(f"Structures remaining: {len(curated):,}")

Generating standardized SMILES representations...


  0%|          | 0/13990 [00:00<?, ?it/s]


SMILES standardization completed.
Structures remaining: 13,990


In [ ]:
# ============================================================
# Step 9. MIC unit recognition and molar standardization
# ============================================================

def normalize_mic_unit(unit):
    """
    Normalize common ChEMBL concentration-unit representations
    into a set of standardized internal unit labels.

    This function does not modify the original Standard Units column.
    """

    if pd.isna(unit):
        return None

    unit = str(unit).strip()

    # Normalize Unicode micro symbols
    unit = unit.replace("μ", "u")
    unit = unit.replace("µ", "u")

    # Normalize multiplication/dot symbols and minus sign
    unit = unit.replace("·", ".")
    unit = unit.replace("−", "-")

    # Remove spaces
    unit = unit.replace(" ", "")

    unit_aliases = {

        # Molar
        "M": "M",
        "mol/L": "M",
        "mol.L-1": "M",
        "mol.l-1": "M",

        # Millimolar
        "mM": "mM",
        "mmol/L": "mM",
        "mmol.L-1": "mM",
        "mmol.l-1": "mM",

        # Micromolar
        "uM": "uM",
        "umol/L": "uM",
        "umol.L-1": "uM",
        "umol.l-1": "uM",

        # Nanomolar
        "nM": "nM",
        "nmol/L": "nM",
        "nmol.L-1": "nM",
        "nmol.l-1": "nM",

        # Picomolar
        "pM": "pM",
        "pmol/L": "pM",
        "pmol.L-1": "pM",
        "pmol.l-1": "pM",

        # Micrograms per milliliter
        "ug/mL": "ug/mL",
        "ug.mL-1": "ug/mL",
        "ug.ml-1": "ug/mL",

        # mg/L is numerically equivalent to ug/mL
        "mg/L": "ug/mL",
        "mg.L-1": "ug/mL",
        "mg.l-1": "ug/mL",

        # Additional mass concentration formats
        "mg/mL": "mg/mL",
        "mg.mL-1": "mg/mL",
        "mg.ml-1": "mg/mL",

        "g/L": "g/L",
        "g.L-1": "g/L",
        "g.l-1": "g/L",

        "ng/mL": "ng/mL",
        "ng.mL-1": "ng/mL",
        "ng.ml-1": "ng/mL"
    }

    return unit_aliases.get(unit)


def convert_mic_to_molar(value, unit, molecular_weight):
    """
    Convert a MIC value to molar concentration (M).

    Parameters
    ----------
    value : float
        Reported Standard Value.

    unit : str
        Reported Standard Units.

    molecular_weight : float
        Molecular weight in g/mol. Required only for
        mass-based concentrations.

    Returns
    -------
    float
        MIC in mol/L (M), or NaN if conversion cannot be performed.
    """

    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan

    if value <= 0:
        return np.nan

    normalized_unit = normalize_mic_unit(unit)

    if normalized_unit is None:
        return np.nan

    # Direct molar conversions
    molar_conversion_factors = {
        "M": 1.0,
        "mM": 1e-3,
        "uM": 1e-6,
        "nM": 1e-9,
        "pM": 1e-12
    }

    if normalized_unit in molar_conversion_factors:

        return (
            value
            * molar_conversion_factors[normalized_unit]
        )

    # Molecular weight is required for mass-based units
    try:
        molecular_weight = float(molecular_weight)
    except (TypeError, ValueError):
        return np.nan

    if (
        not np.isfinite(molecular_weight)
        or molecular_weight <= 0
    ):
        return np.nan

    # ug/mL = mg/L = 1e-3 g/L
    if normalized_unit == "ug/mL":
        return (
            value * 1e-3
        ) / molecular_weight

    # mg/mL = g/L
    if normalized_unit == "mg/mL":
        return value / molecular_weight

    # g/L
    if normalized_unit == "g/L":
        return value / molecular_weight

    # ng/mL = 1e-6 g/L
    if normalized_unit == "ng/mL":
        return (
            value * 1e-6
        ) / molecular_weight

    return np.nan

In [ ]:
# ============================================================
# Step 9B. Apply MIC unit standardization
# ============================================================

print("Unit formats detected in the curated dataset:")
print("=" * 55)

detected_units = sorted(
    curated["Standard Units"]
    .dropna()
    .astype(str)
    .unique()
)

for unit in detected_units:

    normalized = normalize_mic_unit(unit)

    if normalized is None:
        status = "UNSUPPORTED"
    else:
        status = f"recognized as {normalized}"

    print(f"{unit:<25} -> {status}")

# Convert Molecular Weight internally to numeric values
molecular_weight_numeric = pd.to_numeric(
    curated["Molecular Weight"],
    errors="coerce"
)

print("\nCalculating MIC values in molar concentration...")

curated["MIC_M"] = [
    convert_mic_to_molar(
        value,
        unit,
        molecular_weight
    )
    for value, unit, molecular_weight in tqdm(
        zip(
            curated["_Standard_Value_Numeric"],
            curated["Standard Units"],
            molecular_weight_numeric
        ),
        total=len(curated)
    )
]

# Identify records that could not be converted
failed_molar_conversion = (
    curated["MIC_M"].isna()
    | ~np.isfinite(curated["MIC_M"])
    | (curated["MIC_M"] <= 0)
)

audit["MIC unit conversion failures removed"] = int(
    failed_molar_conversion.sum()
)

curated = curated.loc[
    ~failed_molar_conversion
].copy()

print("\nMIC standardization completed.")
print("-" * 55)
print(
    f"Records with valid molar MIC values: "
    f"{len(curated):,}"
)

Unit formats detected in the curated dataset:
nM                        -> recognized as nM
ug.mL-1                   -> recognized as ug/mL

Calculating MIC values in molar concentration...


  0%|          | 0/13990 [00:00<?, ?it/s]


MIC standardization completed.
-------------------------------------------------------
Records with valid molar MIC values: 13,990


In [ ]:
# ============================================================
# Step 10. Duplicate structure resolution
# ============================================================

records_before_deduplication = len(curated)

# Count rows belonging to duplicated canonical structures
duplicate_row_mask = curated.duplicated(
    subset=["Canonical_SMILES"],
    keep=False
)

duplicate_rows = int(
    duplicate_row_mask.sum()
)

duplicate_structures = int(
    curated.loc[
        duplicate_row_mask,
        "Canonical_SMILES"
    ].nunique()
)

print("DUPLICATE STRUCTURE ANALYSIS")
print("=" * 55)
print(
    f"Records before duplicate removal : "
    f"{records_before_deduplication:,}"
)
print(
    f"Rows involved in duplicates      : "
    f"{duplicate_rows:,}"
)
print(
    f"Unique duplicated structures     : "
    f"{duplicate_structures:,}"
)

# Sort by canonical structure and molar MIC.
# Lowest MIC appears first within each structural group.
curated = curated.sort_values(
    by=[
        "Canonical_SMILES",
        "MIC_M"
    ],
    ascending=[
        True,
        True
    ],
    kind="stable"
)

# Retain the record with the lowest MIC_M
curated = curated.drop_duplicates(
    subset=["Canonical_SMILES"],
    keep="first"
).copy()

audit["Duplicate records removed"] = (
    records_before_deduplication
    - len(curated)
)

print("-" * 55)
print(
    f"Records after duplicate removal  : "
    f"{len(curated):,}"
)

DUPLICATE STRUCTURE ANALYSIS
Records before duplicate removal : 13,990
Rows involved in duplicates      : 8,902
Unique duplicated structures     : 1,375
-------------------------------------------------------
Records after duplicate removal  : 6,463


In [ ]:
# ============================================================
# Step 11. pMIC calculation
# ============================================================

curated["pMIC"] = -np.log10(
    curated["MIC_M"].astype(float)
)

if curated["pMIC"].isna().any():
    raise ValueError(
        "Unexpected missing pMIC values were generated."
    )

print("pMIC calculation completed.")
print("-" * 55)

print(
    f"Minimum pMIC : "
    f"{curated['pMIC'].min():.4f}"
)

print(
    f"Maximum pMIC : "
    f"{curated['pMIC'].max():.4f}"
)

print(
    f"Mean pMIC    : "
    f"{curated['pMIC'].mean():.4f}"
)

pMIC calculation completed.
-------------------------------------------------------
Minimum pMIC : 0.3615
Maximum pMIC : 9.5300
Mean pMIC    : 4.4932


In [ ]:
# ============================================================
# Step 12. Final dataset organization
# ============================================================

NEW_COLUMNS = [
    "SMILES_Clean",
    "Canonical_SMILES",
    "Isomeric_SMILES",
    "MIC_M",
    "pMIC"
]

# Keep original ChEMBL columns while avoiding duplicate names
base_columns = [
    column
    for column in original_columns
    if column not in NEW_COLUMNS
]

# Locate Molecule Name
if "Molecule Name" not in base_columns:
    raise ValueError(
        "'Molecule Name' column was not found."
    )

insert_position = (
    base_columns.index("Molecule Name") + 1
)

# Insert newly calculated columns immediately after Molecule Name
final_column_order = (
    base_columns[:insert_position]
    + NEW_COLUMNS
    + base_columns[insert_position:]
)

final_dataset = curated[
    final_column_order
].copy()

# Restore a clean sequential index
final_dataset.reset_index(
    drop=True,
    inplace=True
)

print("Final dataset successfully organized.")
print("-" * 55)
print(
    f"Final records : {len(final_dataset):,}"
)
print(
    f"Final columns : {final_dataset.shape[1]}"
)

print("\nFirst columns in the final dataset:")
print(
    final_dataset.columns[:12].tolist()
)

Final dataset successfully organized.
-------------------------------------------------------
Final records : 6,463
Final columns : 53

First columns in the final dataset:
['Molecule ID', 'Molecule Name', 'SMILES_Clean', 'Canonical_SMILES', 'Isomeric_SMILES', 'MIC_M', 'pMIC', 'Molecule Max Phase', 'Molecular Weight', '#RO5 Violations', 'AlogP', 'Compound Key']


In [ ]:
# ============================================================
# Step 13. Final quality-control summary
# ============================================================

audit["Final unique structures"] = len(
    final_dataset
)

print("FINAL CURATION SUMMARY")
print("=" * 60)
print(f"Antibacterial model: {MODEL_NAME}")
print("=" * 60)

for key, value in audit.items():
    print(
        f"{key:<42}: {value:,}"
    )

print("=" * 60)

print(
    f"Final pMIC range: "
    f"{final_dataset['pMIC'].min():.4f} "
    f"to "
    f"{final_dataset['pMIC'].max():.4f}"
)

FINAL CURATION SUMMARY
Antibacterial model: Acinetobacter baumannii
Initial records                           : 14,617
Non-MIC records removed                   : 185
Missing SMILES removed                    : 52
Missing Standard Value removed            : 389
Non-positive Standard Value removed       : 1
Missing Standard Units removed            : 0
Invalid/metal structures removed          : 0
SMILES standardization failures removed   : 0
MIC unit conversion failures removed      : 0
Duplicate records removed                 : 7,527
Final unique structures                   : 6,463
Final pMIC range: 0.3615 to 9.5300


In [ ]:
# ============================================================
# Step 14. Export final curated dataset
# ============================================================

OUTPUT_FILENAME = (
    f"{MODEL_PREFIX}_ChEMBL_MIC_Curated.csv"
)

final_dataset.to_csv(
    OUTPUT_FILENAME,
    index=False
)

print("Final dataset successfully exported.")
print("-" * 55)
print(f"Output file: {OUTPUT_FILENAME}")
print(
    f"Records    : {len(final_dataset):,}"
)
print(
    f"Columns    : {final_dataset.shape[1]}"
)

# Download the final CSV file to the local computer
files.download(OUTPUT_FILENAME)

Final dataset successfully exported.
-------------------------------------------------------
Output file: A_baumannii_ChEMBL_MIC_Curated.csv
Records    : 6,463
Columns    : 53


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>